# 07 - Retrieval ablation: dense, keyword (BM25) and hybrid search

Every combination of **search type** (dense / BM25 / hybrid), **company-year filter** (off / on) and
**reranker** (off / on) on the 155 answerable questions, text chunks, local reranker (bge-reranker-v2-m3). No
language model and no API: only retrieval, so it is cheap. Metrics: hit@1, hit@5, all@5, all@8, all@10, recall@5, MRR and
R-precision.

All searches give 30 candidates per list (BM25 alone: 30), so the combinations differ only in what they are meant to
differ in. Without a reranker the hybrid lists are interleaved. Needs the Colab secret `HF_TOKEN` and a T4 GPU.
Notebook 03 stays as it was; its three configurations are a subset of this grid.

In [ ]:
import os
import subprocess
import sys

from google.colab import userdata

REPO = "10K-agentic-rag"
BRANCH = "main"
try:  # only needed while the repository is private (a read-only token for this one repository)
    token = userdata.get("GITHUB_TOKEN")
except Exception:  # noqa: BLE001  the secret does not exist: the repository is public
    token = None
login = f"{token}@" if token else ""
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "-b", BRANCH, f"https://{login}github.com/ilaria-cesa/{REPO}.git"],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
os.chdir(REPO)
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements.txt

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(torch.cuda.get_device_name(0))

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # never hard-code the token
HF_REPO = "IlariaCesa/10k-agentic-rag-data"

In [ ]:
import numpy as np

from src.eval.golden import load_golden, validate
from src.hub.dataset import download_folder
from src.retrieval.embed import embed_texts, load_chunks

HF_REPO = "IlariaCesa/10k-agentic-rag-data"
download_folder(HF_REPO, "text", Path("data"))
chunks = load_chunks("data/text/chunks_1600.jsonl")
vectors = np.load("data/text/embeddings_bge-m3.npy").astype(np.float32)
assert len(chunks) == len(vectors)
all_questions = load_golden(Path("questions/questions.jsonl"))
errors, _ = validate(all_questions, chunks)
assert not errors, errors
answerable = [q for q in all_questions if q["expected"] == "answer"]
print(len(answerable), "answerable questions")

In [ ]:
from sentence_transformers import SentenceTransformer

from src.eval.ablation import grid, run_ablation
from src.retrieval.pipeline import Corpus
from src.retrieval.rerank import Reranker

model = SentenceTransformer("BAAI/bge-m3", device="cuda", model_kwargs={"dtype": torch.float16})
model.max_seq_length = 1024
query_vectors = embed_texts(model, [q["question"] for q in answerable], batch_size=32)
corpus = Corpus(chunks, vectors)
reranker = Reranker(max_length=1024)
print(len(grid()), "combinations")
rows = run_ablation(corpus, answerable, query_vectors, reranker)
df = pd.DataFrame(rows)
os.makedirs("results", exist_ok=True)
df.to_json("results/retrieval_ablation.jsonl", orient="records", lines=True)

In [ ]:
pd.set_option("display.width", 200)
cols = ["hit@1", "hit@5", "all@5", "all@8", "all@10", "recall@5", "mrr", "r_precision"]
order = [c.name for c in grid()]
table = df.groupby("config")[cols].mean().reindex(order).round(3)
print(table.to_string())
for split in ["dev", "test", "agentic"]:
    part = df[df["split"] == split]
    print("\n" + split, "(", part["id"].nunique(), "questions )  hit@1 / all@5 / all@10")
    print(
        part.groupby("config")[["hit@1", "all@5", "all@10"]]
        .mean()
        .reindex(order)
        .round(3)
        .to_string()
    )

In [ ]:
from src.eval.grades_report import sign_test

PAIRS = [  # (a, b): is a better than b?
    ("dense + filter", "dense"),
    ("dense + rerank", "dense"),
    ("dense + filter + rerank", "dense + filter"),
    ("bm25", "dense"),
    ("bm25 + filter + rerank", "dense + filter + rerank"),
    ("hybrid", "dense"),
    ("hybrid + filter", "dense + filter"),
    ("hybrid + filter + rerank", "dense + filter + rerank"),
    ("hybrid + filter + rerank", "bm25 + filter + rerank"),
]
for metric in ["hit@1", "all@5", "all@10"]:
    wide = df.pivot(index="id", columns="config", values=metric)
    print(metric, "(a better / b better, sign test p)")
    for a, b in PAIRS:
        better = int((wide[a] > wide[b]).sum())
        worse = int((wide[a] < wide[b]).sum())
        print(f"  {a:26s} vs {b:26s} {better:3d} / {worse:3d}  p={sign_test(better, worse):.3f}")

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("retrieval_ablation", "zip", "results")
files.download("retrieval_ablation.zip")